# **Homework Assignment QT5: Protein Sequence Embeddings**

**Name:** Vedasravas Dasari

**M ID:** M16446787

**Colab Link:** https://colab.research.google.com/drive/1kLPbwuduwnq3l536zVOSX7oT9AqqX9m7?usp=sharing

**Submission Date:** 02/14/2025

---

**Course:** CS7099-Introduction to Bioinformatics

**Instructor:** Jarek Meller

---

All results, figures, and discussion are included below.

In [ ]:
# The script is to test ESM-2 fine tuning to classify sequences.
# The code is based on the tutorial at
# https://github.com/huggingface/notebooks/blob/main/examples/protein_language_modeling.ipynb

In [ ]:
# Install necessary libraries
# update pip
!pip3 install -U pip > /dev/null

!pip install transformers evaluate datasets requests pandas scikit-learn > /dev/null

In [ ]:
# Define a request to UniProt RESTful API to retrieve annotated sequence
# The URL can be formed by the Advanced Search interafce at UniProt
# Original URL from the tutorial (invalid)
# https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession,sequence,cc_subcellular_location&format=tsv&query=((organism_id:9606) AND (reviewed:true) AND (length:[80 TO 500]))
# Made through UniProt web-interface on March 19, 2025
# https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Cprotein_name%2Cgene_names%2Ccc_subcellular_location%2Csequence&format=tsv&query=%28%28organism_id%3A9606%29+AND+%28reviewed%3Atrue%29+AND+%28length%3A%5B80+TO+500%5D%29%29
# which translates to the following
# https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession,protein_name,gene_names,cc_subcellular_location,sequence&format=tsv&query=((organism_id:9606) AND (reviewed:true) AND (length:[80 TO 500]))

query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Cprotein_name%2Cgene_names%2Ccc_subcellular_location%2Csequence&format=tsv&query=%28%28organism_id%3A9606%29+AND+%28reviewed%3Atrue%29+AND+%28length%3A%5B80+TO+500%5D%29%29"


In [ ]:
# Download the human proteins from UniProt, gzipped in tsv format
import requests
uniprot_request = requests.get(query_url)

In [ ]:
# To get this data into Pandas, we use a BytesIO object, which Pandas will treat like a file.
# If you downloaded the data as a file you can skip this bit and just pass the filepath directly to read_csv.
from io import BytesIO
import pandas

bio = BytesIO(uniprot_request.content)

df = pandas.read_csv(bio, compression='gzip', sep='\t')
df

,Entry,Protein names,Gene Names,Subcellular location [CC],Sequence
0,A0A0K2S4Q6,Protein CD300H (CD300 antigen-like family memb...,CD300H,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...
1,A0AVI4,E3 ubiquitin-protein ligase TM129 (EC 2.3.2.27...,TMEM129,SUBCELLULAR LOCATION: Endoplasmic reticulum me...,MDSPEVTFTLAYLVFAVCFVFTPNEFHAAGLTVQNLLSGWLGSEDA...
2,A0JLT2,Mediator of RNA polymerase II transcription su...,MED19 LCMR1,SUBCELLULAR LOCATION: Nucleus {ECO:0000305}.,MENFTALFGAQADPPPPPTALGFGPGKPPPPPPPPAGGGPGTAPPP...
3,A0M8Q6,Immunoglobulin lambda constant 7 (Ig lambda-7 ...,IGLC7,SUBCELLULAR LOCATION: Secreted {ECO:0000303|Pu...,GQPKAAPSVTLFPPSSEELQANKATLVCLVSDFNPGAVTVAWKADG...
4,A0PJY2,Fez family zinc finger protein 1 (Zinc finger ...,FEZF1 FEZ ZNF312B,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...
...,...,...,...,...,...
11972,Q9H8W2,Putative uncharacterized protein encoded by LI...,LINC00472 C6orf155,NaN,MRPGSSPRAPECGAPALPRPQLDRLPARPAPSRGRGAPSLRWPAKE...
11973,Q9HAA7,Putative uncharacterized protein FLJ11871,NaN,NaN,MLFGIRILVNTPSPLVTGLHHYNPSIHRDQGECANQWRKGPGSAHL...
11974,Q9NZ38,Uncharacterized protein IDI2-AS1 (IDI2 antisen...,IDI2-AS1 C10orf110 HT009,NaN,MAFPGQSDTKMQWPEVPALPLLSSLCMAMVRKSSALGKEVGRRSEG...
11975,Q9UFV3,Putative uncharacterized protein DKFZp434L187,NaN,NaN,MAETYRRSRQHEQLPGQRHMDLLTGYSKLIQSRLKLLLHLGSQPPV...


In [ ]:
# Drop proteins with missing columns
df = df.dropna()
df

,Entry,Protein names,Gene Names,Subcellular location [CC],Sequence
0,A0A0K2S4Q6,Protein CD300H (CD300 antigen-like family memb...,CD300H,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...
1,A0AVI4,E3 ubiquitin-protein ligase TM129 (EC 2.3.2.27...,TMEM129,SUBCELLULAR LOCATION: Endoplasmic reticulum me...,MDSPEVTFTLAYLVFAVCFVFTPNEFHAAGLTVQNLLSGWLGSEDA...
2,A0JLT2,Mediator of RNA polymerase II transcription su...,MED19 LCMR1,SUBCELLULAR LOCATION: Nucleus {ECO:0000305}.,MENFTALFGAQADPPPPPTALGFGPGKPPPPPPPPAGGGPGTAPPP...
3,A0M8Q6,Immunoglobulin lambda constant 7 (Ig lambda-7 ...,IGLC7,SUBCELLULAR LOCATION: Secreted {ECO:0000303|Pu...,GQPKAAPSVTLFPPSSEELQANKATLVCLVSDFNPGAVTVAWKADG...
4,A0PJY2,Fez family zinc finger protein 1 (Zinc finger ...,FEZF1 FEZ ZNF312B,SUBCELLULAR LOCATION: Nucleus {ECO:0000269|Pub...,MDSSCHNATTKMLATAPARGNMMSTSKPLAFSIERIMARTPEPKAL...
...,...,...,...,...,...
11895,Q86UQ5,Gilles de la Tourette syndrome chromosomal reg...,GTSCR1,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MQSDIYHPGHSFPSWVLCWVHSCGHEGHLRETAEIRKTHQNGDLQI...
11918,Q8N8V8,Transmembrane protein 105,TMEM105,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MLLKVRRASLKPPATPHQGAFRAGNVIGQLIYLLTWSLFTAWLRPP...
11955,Q96N68,Putative uncharacterized protein C18orf15,C18orf15,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MQGQGALKESHIHLPTEQPEASLVLQGQLAESSALGPKGALRPQAQ...
11963,Q9H0A3,Transmembrane protein 191A,TMEM191A,SUBCELLULAR LOCATION: Membrane {ECO:0000255}; ...,MMNNTDFLMLNNPWNKLCLVSMDFCFPLDFVSNLFWIFASKFIIVT...


In [ ]:
# Now we'll make one dataframe of proteins that contain cytosol or cytoplasm in their subcellular localization column,
# and a second that mentions the membrane or cell membrane.
# To ensure we don't get overlap, we ensure each dataframe only contains proteins that don't match the other search term.

cytosolic = df['Subcellular location [CC]'].str.contains("Cytosol") | df['Subcellular location [CC]'].str.contains("Cytoplasm")
membrane = df['Subcellular location [CC]'].str.contains("Membrane") | df['Subcellular location [CC]'].str.contains("Cell membrane")

In [ ]:
cytosolic_df = df[cytosolic & ~membrane]
cytosolic_df

,Entry,Protein names,Gene Names,Subcellular location [CC],Sequence
9,A1E959,Odontogenic ameloblast-associated protein (Apin),ODAM APIN,SUBCELLULAR LOCATION: Secreted {ECO:0000250|Un...,MKIIILLGFLGATLSAPLIPQRLMSASNSNELLLNLNNGQLLPLQL...
14,A1XBS5,CBY1-interacting BAR domain-containing protein 1,CIBAR1 FAM92A FAM92A1,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000269|P...,MMRRTLENRNAQTKQLQTAVSNVEKHFGELCQIFAAYVRKTARLRD...
18,A2RU49,Hydroxylysine kinase (5-hydroxy-L-lysine kinas...,HYKK AGPHD1,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000305}.,MSSGNYQQSEALSKPTFSEEQASALVESVFGLKVSKVRPLPSYDDQ...
20,A2RUH7,Myosin-binding protein H-like,MYBPHL,"SUBCELLULAR LOCATION: Cytoplasm, myofibril, sa...",MEAATAPEVAAGSKLKVKEASPADAEPPQASPGQGAGSPTPQLLPP...
21,A4D126,D-ribitol-5-phosphate cytidylyltransferase (EC...,CRPPA ISPD,"SUBCELLULAR LOCATION: Cytoplasm, cytosol {ECO:...",MEAGPPGSARPAEPGPCLSGQRGADHTASASLQSVAGTEPGRHPQA...
...,...,...,...,...,...
11495,Q8NBC4,Uncharacterized protein C20orf203,C20orf203,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000269|P...,MFPRPVLNSRAQAILLPQPPNMLDHRQWPPRLASFPFTKTGMLSRA...
11535,Q8TDY3,Actin-related protein T2 (ARP-T2) (Actin-relat...,ACTRT2 ARPM2,"SUBCELLULAR LOCATION: Cytoplasm, cytoskeleton ...",MFNPHALDSPAVIFDNGSGFCKAGLSGEFGPRHMVSSIVGHLKFQA...
11547,Q8WWF8,Calcyphosin-like protein,CAPSL,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000305}.,MAGTARHDREMAIQAKKKLTTATDPIERLRLQCLARGSAGIKGLGR...
11671,Q9NUJ7,PI-PLC X domain-containing protein 1,PLCXD1,SUBCELLULAR LOCATION: Cytoplasm {ECO:0000269|P...,MGGQVSASNSFSRLHCRNANEDWMSALCPRLWDVPLHHLSIPGSHD...


In [ ]:
membrane_df = df[membrane & ~cytosolic]
membrane_df

,Entry,Protein names,Gene Names,Subcellular location [CC],Sequence
0,A0A0K2S4Q6,Protein CD300H (CD300 antigen-like family memb...,CD300H,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...,MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...
3,A0M8Q6,Immunoglobulin lambda constant 7 (Ig lambda-7 ...,IGLC7,SUBCELLULAR LOCATION: Secreted {ECO:0000303|Pu...,GQPKAAPSVTLFPPSSEELQANKATLVCLVSDFNPGAVTVAWKADG...
17,A2RU14,Transmembrane protein 218,TMEM218,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MAGTVLGVGAGVFILALLWVAVLLLCVLLSRASGAARFSVIFLFFG...
33,A5X5Y0,5-hydroxytryptamine receptor 3E (5-HT3-E) (5-H...,HTR3E,SUBCELLULAR LOCATION: Postsynaptic cell membra...,MEGSWFHRKRFSFYLLLGFLLQGRGVTFTINCSGFGQHGADPTALN...
36,A6ND01,Sperm-egg fusion protein Juno (Folate receptor...,IZUMO1R FOLR4 JUNO,SUBCELLULAR LOCATION: Cell membrane {ECO:00002...,MACWWPLLLELWTVMPTWAGDELLNICMNAKHHKRVPSPEDKLYEE...
...,...,...,...,...,...
11895,Q86UQ5,Gilles de la Tourette syndrome chromosomal reg...,GTSCR1,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MQSDIYHPGHSFPSWVLCWVHSCGHEGHLRETAEIRKTHQNGDLQI...
11918,Q8N8V8,Transmembrane protein 105,TMEM105,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MLLKVRRASLKPPATPHQGAFRAGNVIGQLIYLLTWSLFTAWLRPP...
11955,Q96N68,Putative uncharacterized protein C18orf15,C18orf15,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...,MQGQGALKESHIHLPTEQPEASLVLQGQLAESSALGPKGALRPQAQ...
11963,Q9H0A3,Transmembrane protein 191A,TMEM191A,SUBCELLULAR LOCATION: Membrane {ECO:0000255}; ...,MMNNTDFLMLNNPWNKLCLVSMDFCFPLDFVSNLFWIFASKFIIVT...


In [ ]:
# Now, let's make a list of sequences from each df and generate the associated labels.
# We'll use 0 as the label for cytosolic proteins and 1 as the label for membrane proteins.

cytosolic_sequences = cytosolic_df["Sequence"].tolist()
cytosolic_labels = [0 for protein in cytosolic_sequences]

membrane_sequences = membrane_df["Sequence"].tolist()
membrane_labels = [1 for protein in membrane_sequences]

# Now, we can concatenate these lists together to get the sequences and labels lists that will form our final training data.

sequences = cytosolic_sequences + membrane_sequences
labels = cytosolic_labels + membrane_labels

len(sequences) == len(labels)

True

In [ ]:
# Since the data we're loading isn't prepared for us as a machine learning dataset, we'll have to split the data into train and test sets ourselves.

from sklearn.model_selection import train_test_split

# train_sequences, test_sequences, train_labels, test_labels = train_test_split(sequences, labels, test_size=0.25, shuffle=True)

# Step 1: Split into 75% (train+val) and 25% (test)
train_val_sequences, test_sequences, train_val_labels, test_labels = train_test_split(
    sequences, labels, test_size=0.25, shuffle=True, random_state=42
)

# Step 2: Split the 75% into 60% (train) and 15% (validation)
train_sequences, val_sequences, train_labels, val_labels = train_test_split(
    train_val_sequences, train_val_labels, test_size=0.2, shuffle=True, random_state=42
)

In [ ]:
# Quick check to make sure we got it right

print("Total sequences=", len(sequences))
print("Cytoplasmic=", labels.count(0), "(", round(labels.count(0) / len(sequences) * 100, 2), "%)", "Membrane=", labels.count(1), "(", round(labels.count(1) / len(sequences) * 100, 2), "%)")
print("Subsets: Train=", len(train_sequences), "Validation=", len(val_sequences), "Test=", len(test_sequences))


Total sequences= 5171
Cytoplasmic= 2643 ( 51.11 %) Membrane= 2528 ( 48.89 %)
Subsets: Train= 3102 Validation= 776 Test= 1293


In [ ]:
# There are several ESM-2 checkpoints with differing model sizes.
# Larger models will generally have better accuracy, but they require more GPU memory and will take much longer to train.
# The available ESM-2 checkpoints (at time of writing) are:
# Checkpoint name 	Num layers 	Num parameters
# esm2_t48_15B_UR50D 	48 	15B
# esm2_t36_3B_UR50D 	36 	3B
# esm2_t33_650M_UR50D 	33 	650M
# esm2_t30_150M_UR50D 	30 	150M
# esm2_t12_35M_UR50D 	12 	35M
# esm2_t6_8M_UR50D 	6 	8M

# Note that the larger checkpoints may be very difficult to train without a large cloud GPU like an A100 or H100,
# and the largest 15B parameter checkpoint will probably be impossible to train on any single GPU!
# Also, note that memory usage for attention during training will scale as O(batch_size * num_layers * seq_len^2),
# so larger models on long sequences will use quite a lot of memory!
# We will use the esm2_t12_35M_UR50D checkpoint for this notebook, which should train on any Colab instance or modern GPU.

# model_checkpoint = "facebook/esm2_t12_35M_UR50D"
model_checkpoint = "facebook/esm2_t6_8M_UR50D"

In [ ]:
# Tokenizing the data
# All inputs to neural nets must be numerical. The process of converting strings into numerical indices suitable for a neural net is called tokenization.
# For natural language this can be quite complex, as usually the network's vocabulary will not contain every possible word,
# which means the tokenizer must handle splitting rarer words into pieces, as well as all the complexities of capitalization and unicode characters and so on.

# With proteins, however, things are very easy. In protein language models, each amino acid is converted to a single token.
# Every model on transformers comes with an associated tokenizer that handles tokenization for it, and protein language models are no different. Let's get our tokenizer!

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
# Check point - check the result on a first sequence
test_tokenizer = tokenizer(train_sequences[0])

print("Sequence=", train_sequences[0])
print("Length=", len(train_sequences[0]), "Tokens=", len(test_tokenizer.input_ids), "Masks=", len(test_tokenizer.attention_mask))
print(test_tokenizer)


Sequence= MFLSSRMITSVSPSTSTNSSFLLTGFSGMEQQYPWLSIPFSSIYAMVLLGNCMVLHVIWTEPSLHQPMFYFLSMLALTDLCMGLSTVYTVLGILWGIIREISLDSCIAQSYFIHGLSFMESSVLLTMAFDRYIAICNPLRYSSILTNSRIIKIGLTIIGRSFFFITPPIICLKFFNYCHFHILSHSFCLHQDLLRLACSDIRFNSYYALMLVICILLLDAILILFSYILILKSVLAVASQEERHKLFQTCISHICAVLVFYIPIISLTMVHRFGKHLSPVAHVLIGNIYILFPPLMNPIIYSVKTQQIHTRMLRLFSLKRY
Length= 321 Tokens= 323 Masks= 323
{'input_ids': [0, 20, 18, 4, 8, 8, 10, 20, 12, 11, 8, 7, 8, 14, 8, 11, 8, 11, 17, 8, 8, 18, 4, 4, 11, 6, 18, 8, 6, 20, 9, 16, 16, 19, 14, 22, 4, 8, 12, 14, 18, 8, 8, 12, 19, 5, 20, 7, 4, 4, 6, 17, 23, 20, 7, 4, 21, 7, 12, 22, 11, 9, 14, 8, 4, 21, 16, 14, 20, 18, 19, 18, 4, 8, 20, 4, 5, 4, 11, 13, 4, 23, 20, 6, 4, 8, 11, 7, 19, 11, 7, 4, 6, 12, 4, 22, 6, 12, 12, 10, 9, 12, 8, 4, 13, 8, 23, 12, 5, 16, 8, 19, 18, 12, 21, 6, 4, 8, 18, 20, 9, 8, 8, 7, 4, 4, 11, 20, 5, 18, 13, 10, 19, 12, 5, 12, 23, 17, 14, 4, 10, 19, 8, 8, 12, 4, 11, 17, 8, 10, 12, 12, 15, 12, 6, 4, 11, 12, 12, 6, 10, 8, 18, 18, 18, 12, 11, 14, 14, 12, 12, 23, 4, 

In [ ]:
# Tokenize our whole dataset. Note that we don't need to do anything with the labels, as they're already in the format we need.
train_tokenized = tokenizer(train_sequences)
val_tokenized = tokenizer(val_sequences)
test_tokenized = tokenizer(test_sequences)

In [ ]:
# Dataset creation

# Now we want to turn this data into a dataset that PyTorch can load samples from.
# We can use the HuggingFace Dataset class for this, although if you prefer you can also use torch.utils.data.Dataset, at the cost of some more boilerplate code.

from datasets import Dataset
train_dataset = Dataset.from_dict(train_tokenized)
val_dataset = Dataset.from_dict(val_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3102
})

In [ ]:
# Add labels as an extra column to the datasets.

train_dataset = train_dataset.add_column("labels", train_labels)
val_dataset = val_dataset.add_column("labels", val_labels)
test_dataset = test_dataset.add_column("labels", test_labels)
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3102
})

In [ ]:
# Model loading

# Next, we want to load our model. Make sure to use exactly the same model as you used when loading the tokenizer, or your model might not understand the tokenization scheme you're using!

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = max(train_labels + val_labels + test_labels) + 1  # Add 1 since 0 can be a label
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# These warnings are telling us that the model is discarding some weights that it used for language modelling (the lm_head) and adding some weights for sequence classification (the classifier).
# This is exactly what we expect when we want to fine-tune a language model on a sequence classification task!

In [ ]:
# define the metric we will use to evaluate our models and write a compute_metrics function. We can load this from the evaluate library.

from evaluate import load
import numpy as np

metric = load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)



In [ ]:
# Set Up the Trainer for Evaluation Only - to see the performance of the pre-tained model

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

eval_args = TrainingArguments(
    per_device_eval_batch_size=8,
    do_train=False,
    do_eval=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics  # Your custom accuracy function using `evaluate`
)


In [ ]:
# Make predictions on the blind test set
predictions = trainer.predict(test_dataset)
logits = predictions.predictions
true_labels = test_dataset["labels"]

# Convert logits to predicted class indices
predicted_labels = np.argmax(logits, axis=1)

# Use `evaluate` to compute accuracy
from evaluate import load
metric = load("accuracy")
accuracy = metric.compute(predictions=predicted_labels, references=true_labels)

print(f"Pretrained Model Accuracy (no fine-tuning): {accuracy['accuracy']:.4f}")


Pretrained Model Accuracy (no fine-tuning): 0.4888


In [ ]:
# Initialize TrainingArguments. These control the various training hyperparameters, and will be passed to our Trainer.

model_name = model_checkpoint.split("/")[-1]
batch_size = 8

args = TrainingArguments(
    f"{model_name}-finetuned-localization",  # Output directory for model checkpoints
    eval_strategy="epoch",                   # Evaluate after every epoch
    save_strategy="epoch",                   # Save model checkpoint after every epoch
    learning_rate=2e-5,                      # Learning rate (how fast the model updates weights)
    per_device_train_batch_size=batch_size,  # Batch size for training
    per_device_eval_batch_size=batch_size,   # Batch size for evaluation
    num_train_epochs=3,                      # Number of times the model sees the entire dataset
    weight_decay=0.01,                       # Regularization to prevent overfitting
    load_best_model_at_end=True,             # Load best model based on metric at the end
    metric_for_best_model="accuracy",        # Use accuracy to determine the best model
    push_to_hub=False,                       # Do not upload to Hugging Face
    report_to="none"                         # Do not log to Hugging Face
)


# training_args = TrainingArguments(
#     output_dir="./esm_checkpoints",  # Save checkpoints locally
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=3,
#     logging_dir="./logs",
#     push_to_hub=False,  # Prevent pushing to Hugging Face
#     report_to="none",  # Disable logging to Hugging Face
# )


In [ ]:
# initialize Trainer
# A data collator is a utility in Hugging Face Transformers that helps with batch preparation, particularly:
# Padding, Tensor conversion, Label formatting
# It’s used with a DataLoader or Trainer to turn a list of examples (from a dataset) into a batch that can be passed to a model.

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
# We can now finetune our model by just calling the train method

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Store the trained model
# trained_model = trainer.model  # Saves the trained model in a variable

# Make predictions on the blind test set
predictions = trainer.predict(test_dataset)


Epoch,Training Loss,Validation Loss


In [ ]:
# Extract logits (raw model outputs)
logits = predictions.predictions

# Convert logits to class predictions
predicted_labels = np.argmax(logits, axis=1)

# Get true labels from test set
true_labels = test_dataset["labels"]

# Compute accuracy
accuracy = metric.compute(predictions=predicted_labels, references=true_labels)

# Print the accuracy
print(f"Test Accuracy: {accuracy['accuracy']:.2f}")

Test Accuracy: 0.93
